# Контрольные точки: public → hard v2, старая → новая модель

**Эксперимент, не развёртывание.** Старая модель — неизменённый локальный артефакт `/srv/Hakaton/models/baseline/tfidf_baseline.joblib` (5 категорий 102–106, пороги 0.5). Новая — тот же TF-IDF + OneVsRest SGDClassifier, обученный на **public train + hard v2 train**, с шестью категориями 101–106 и порогами 0.5. Синтетика получает равный вес на запись; это один базовый эксперимент, а не оптимальная смесь.

Восемь точек вычисляются на одинаковых для обеих моделей public validation (729), public test (208), hard v2 validation (2440) и hard v2 test (3016). Public test имеет повторяющиеся описания с противоречивыми метками и пересечение с обучающими объявлениями: его результат нельзя считать независимой оценкой. Public validation содержит слабые автоматические метки; hard v2 — синтетический корпус. V2 validation здесь **только измеряется**, а не используется для подбора порогов; v2 test уже просмотрен и не должен использоваться для настройки модели. Не интерпретировать разницу между корпусами как эффект обучения.

**Запуск:** Python 3.12 и зависимости из `ml-service/app/requirements.txt` и `datasets/requirements-notebook.txt` (для Parquet требуется `pyarrow`); исполнение на сервере с доступом к `/srv/Hakaton/data`, `/srv/Hakaton/models`. Выходной экспериментальный артефакт сохраняется вне Git и не заменяет рабочую модель. Предсказания соответствуют классификации API: `description`, `mcId`, исключение исходной категории, `generate_text=false`.


In [1]:
import hashlib, json
from pathlib import Path
from collections import Counter
import joblib
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.linear_model import SGDClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import precision_recall_fscore_support
import sklearn
import pyarrow.parquet as pq

ROOT = Path('/srv/Hakaton')
REPO = ROOT / 'innopolis_2026'
PUBLIC = ROOT / 'data/processed'
V2 = REPO / 'datasets/hard_dataset/v2'
OLD_MODEL = ROOT / 'models/baseline/tfidf_baseline.joblib'
EXPERIMENT_MODEL = ROOT / 'models/experiments/tfidf_public_plus_v2_six_heads.joblib'
IDS = (101, 102, 103, 104, 105, 106)
SEED = 42
print('sklearn:', sklearn.__version__, '| numpy:', np.__version__)
print('old model:', OLD_MODEL, 'sha256:', hashlib.sha256(OLD_MODEL.read_bytes()).hexdigest())


sklearn: 1.9.1 | numpy: 2.5.4
old model: /srv/Hakaton/models/baseline/tfidf_baseline.joblib sha256: 668c95e2cecdaa2b94f1a82470e6b89fe3ab1808c765539b45291a22a15004bc


## 1. Данные и контроль пересечений

Для v2 обучающие цели — `offered_microcategories`: исходная категория исключается **после** предсказания. `found_microcategories` служат эталоном конечного ответа. Для public train исходная категория всегда 101; подготовленные `labels` — цели 102–106. Используем готовые разбиения, **не** запускаем `prepare_dataset.py` на v2. Проверяем пересечение нормализованных описаний; отсутствие точного совпадения не исключает семантического сходства или общих шаблонов.


In [2]:
def read_jsonl(path):
    with path.open(encoding='utf-8') as stream:
        return [json.loads(line) for line in stream if line.strip()]

def normalized(text):
    return ' '.join(text.casefold().split())

public_train = read_jsonl(PUBLIC / 'train.jsonl')
public_validation = read_jsonl(PUBLIC / 'validation.jsonl')
v2_train = read_jsonl(V2 / 'train.jsonl')
v2_validation = read_jsonl(V2 / 'validation.jsonl')
v2_test = read_jsonl(V2 / 'test.jsonl')
public_names = {102: 'Штукатурные работы', 103: 'Плиточные работы',
                104: 'Сантехника', 105: 'Электрика',
                106: 'Поклейка обоев и малярные работы'}
name_to_id = {name: mid for mid, name in public_names.items()}
public_test = []
for r in pq.read_table(REPO / 'datasets/public/public-test-dataset.parquet').to_pylist():
    public_test.append({'description': r['description'], 'mcId': 101,
                        'labels': [name_to_id[name] for name in r['categories']]})
assert all(set(r['labels']) <= set(IDS[1:]) for r in public_train + public_validation)
assert all(r['mcId'] in IDS for r in v2_train + v2_validation + v2_test)
assert len(public_train) == 3398 and len(public_validation) == 729
assert len(public_test) == 208
assert len(v2_train) == 8464 and len(v2_validation) == 2440 and len(v2_test) == 3016

train_texts = {normalized(r['description']) for r in public_train + v2_train}
for name, rows in [('public validation', public_validation), ('v2 validation', v2_validation), ('v2 test', v2_test)]:
    overlap = sum(normalized(r['description']) in train_texts for r in rows)
    print(f'{name}: rows={len(rows)}, exact normalized overlap with combined train={overlap}')
    assert overlap == 0, 'Text leakage: inspect datasets before evaluation'
print('public train:', len(public_train), '| v2 train:', len(v2_train))
test_overlap = sum(normalized(r['description']) in train_texts for r in public_test)
print('public test:', len(public_test), '| overlap with combined train:', test_overlap,
      '(public test is not an independent holdout; labels can conflict)')


public validation: rows=729, exact normalized overlap with combined train=0
v2 validation: rows=2440, exact normalized overlap with combined train=0
v2 test: rows=3016, exact normalized overlap with combined train=0
public train: 3398 | v2 train: 8464
public test: 208 | overlap with combined train: 20 (public test is not an independent holdout; labels can conflict)


## 2. Новая модель (не изменяет baseline)

Архитектура и параметры TF-IDF/SGD как в `avito_compare/scripts/train_baseline.py`; дополнительно шестая голова 101. Пороги зафиксированы на 0.5, без подбора по тесту. Историческая модель остаётся нетронутой. Длина обучающих текстов ограничена 6000 символами, как в исходном обучающем скрипте.


In [3]:
train_rows = public_train + v2_train
X = [r['description'][:6000] for r in train_rows]
Y = np.asarray([
    [int(mid in (r['labels'] if 'labels' in r else r['offered_microcategories'])) for mid in IDS]
    for r in train_rows
], dtype=np.int8)
print('train shape:', Y.shape, '| positive counts:', dict(zip(IDS, Y.sum(axis=0).tolist())))

vectorizer = FeatureUnion([
    ('word', TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=18000,
                             sublinear_tf=True, dtype=np.float32)),
    ('char', TfidfVectorizer(analyzer='char', ngram_range=(3, 5), min_df=3,
                             max_features=28000, sublinear_tf=True, dtype=np.float32)),
])
new_pipeline = Pipeline([
    ('vectorizer', vectorizer),
    ('classifier', OneVsRestClassifier(SGDClassifier(loss='log_loss', alpha=0.0001,
                          max_iter=1000, tol=1e-3, random_state=SEED))),
])
new_pipeline.fit(X, Y)
new_model = {'pipeline': new_pipeline, 'ids': IDS, 'thresholds': np.full(len(IDS), 0.5)}
EXPERIMENT_MODEL.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(new_model, EXPERIMENT_MODEL)
print('Experimental model:', EXPERIMENT_MODEL, '| sha256:', hashlib.sha256(EXPERIMENT_MODEL.read_bytes()).hexdigest())


train shape: (11862, 6) | positive counts: {101: 2515, 102: 4266, 103: 4137, 104: 4131, 105: 4125, 106: 4374}
Experimental model: /srv/Hakaton/models/experiments/tfidf_public_plus_v2_six_heads.joblib | sha256: fb32a77c657608f957ecf517a036dbe4fa0a46db34840a22eb1b30b46a29c54c


## 3. Восемь контрольных точек

В каждой строке micro-precision/recall/F1 по всем шести целевым категориям после исключения исходного `mcId`; exact match — доля полностью совпавших наборов. Все метрики — только классификация, не качество `title/text`. Сохранённая старая модель содержит **пять порогов 0.5**, хотя её `metrics.json` также описывает отдельно подобранные пороги.


In [4]:
old_model = joblib.load(OLD_MODEL)  # Только доверенный локально обученный joblib.
assert tuple(old_model['ids']) == IDS[1:]
assert np.array_equal(old_model['thresholds'], np.full(5, 0.5))

def gold_rows(rows):
    return [set(r['labels']) if 'labels' in r else
            {item['mcId'] for item in r['found_microcategories']} for r in rows]

def predict(model, rows):
    probs = model['pipeline'].predict_proba([r['description'] for r in rows])
    return [{int(mid) for mid, p, th in zip(model['ids'], values, model['thresholds'])
             if p >= th and int(mid) != r.get('mcId', 101)}
            for r, values in zip(rows, probs)]

def report(rows, preds):
    gold = gold_rows(rows)
    yt = np.asarray([[int(mid in labels) for mid in IDS] for labels in gold])
    yp = np.asarray([[int(mid in labels) for mid in IDS] for labels in preds])
    p, r, f, _ = precision_recall_fscore_support(yt, yp, average='micro', zero_division=0)
    return {'N': len(rows), 'precision': round(float(p), 4),
            'recall': round(float(r), 4), 'micro_F1': round(float(f), 4),
            'exact_match': round(float(np.all(yt == yp, axis=1).mean()), 4),
            'FP': int(((yp == 1) & (yt == 0)).sum()),
            'FN': int(((yp == 0) & (yt == 1)).sum()),
            'by_category': {
                mid: tuple(round(float(v), 3) for v in
                    precision_recall_fscore_support(yt[:, i], yp[:, i], average='binary',
                                                   zero_division=0)[:3])
                for i, mid in enumerate(IDS)}}

points = {}
for data_name, rows in [('public validation', public_validation), ('public test', public_test),
                        ('hard v2 validation', v2_validation), ('hard v2 test', v2_test)]:
    for model_name, model in [('old', old_model), ('new', new_model)]:
        points[(model_name, data_name)] = report(rows, predict(model, rows))

print(f'{"Модель":<10} {"Данные":<20} {"N":>5} {"Precision":>10} {"Recall":>8} {"Micro-F1":>9} {"Exact":>8} {"FP":>5} {"FN":>5}')
for (model_name, data_name), m in points.items():
    print(f'{model_name:<10} {data_name:<20} {m["N"]:>5} {m["precision"]:>10.4f} {m["recall"]:>8.4f} {m["micro_F1"]:>9.4f} {m["exact_match"]:>8.4f} {m["FP"]:>5} {m["FN"]:>5}')
print('Delta new-old public F1:', round(points[('new','public validation')]['micro_F1'] - points[('old','public validation')]['micro_F1'], 4))
print('Delta new-old v2 F1:', round(points[('new','hard v2 test')]['micro_F1'] - points[('old','hard v2 test')]['micro_F1'], 4))


Модель     Данные                   N  Precision   Recall  Micro-F1    Exact    FP    FN
old        public validation      729     0.9365   0.9334    0.9349   0.8121   116   122
new        public validation      729     0.9281   0.9236    0.9259   0.7805   131   140
old        public test            208     0.7241   0.9254    0.8125   0.6587   104    22
new        public test            208     0.7313   0.8949    0.8049   0.6538    97    31
old        hard v2 validation    2440     0.6685   0.1882    0.2937   0.1557   360  3131
new        hard v2 validation    2440     0.7464   0.6111    0.6720   0.4262   801  1500
old        hard v2 test          3016     0.6124   0.1706    0.2669   0.1280   571  4384
new        hard v2 test          3016     0.6960   0.3855    0.4962   0.2212   890  3248
Delta new-old public F1: -0.009
Delta new-old v2 F1: 0.2293


## 4. Диагностика: категории и исходный mcId

Старая модель не может вернуть 101. Поэтому показываем также срез `source=101`, чтобы не приписывать всё ухудшение лишь отсутствующей голове. Результаты v2 test нужны для описания эксперимента, **не** для итеративной настройки порогов или параметров.


In [5]:
for data_name, rows in [('public validation', public_validation), ('public test', public_test),
                        ('hard v2 validation', v2_validation), ('hard v2 test', v2_test)]:
    print('\n', data_name)
    for model_name, model in [('old', old_model), ('new', new_model)]:
        m = points[(model_name, data_name)]
        print(model_name, 'by-category P/R/F1:', m['by_category'])
        subset = [r for r in rows if r.get('mcId', 101) == 101]
        sub = report(subset, predict(model, subset))
        print('  source=101: N=', len(subset), 'F1=', sub['micro_F1'],
              'exact=', sub['exact_match'])



 public validation
old by-category P/R/F1: {101: (0.0, 0.0, 0.0), 102: (0.925, 0.93, 0.927), 103: (0.905, 0.946, 0.925), 104: (0.94, 0.926, 0.933), 105: (0.951, 0.946, 0.948), 106: (0.961, 0.921, 0.941)}
  source=101: N= 729 F1= 0.9349 exact= 0.8121
new by-category P/R/F1: {101: (0.0, 0.0, 0.0), 102: (0.906, 0.914, 0.91), 103: (0.9, 0.938, 0.918), 104: (0.942, 0.918, 0.93), 105: (0.934, 0.934, 0.934), 106: (0.959, 0.916, 0.937)}
  source=101: N= 729 F1= 0.9259 exact= 0.7805

 public test
old by-category P/R/F1: {101: (0.0, 0.0, 0.0), 102: (0.674, 0.935, 0.784), 103: (0.492, 0.806, 0.611), 104: (0.787, 0.969, 0.869), 105: (0.829, 0.919, 0.872), 106: (0.786, 0.948, 0.859)}
  source=101: N= 208 F1= 0.8125 exact= 0.6587
new by-category P/R/F1: {101: (0.0, 0.0, 0.0), 102: (0.674, 0.935, 0.784), 103: (0.517, 0.861, 0.646), 104: (0.787, 0.969, 0.869), 105: (0.857, 0.811, 0.833), 106: (0.8, 0.897, 0.846)}
  source=101: N= 208 F1= 0.8049 exact= 0.6538

 hard v2 validation
old by-category P/R/F

## Выводы и условия следующей точки

- Сравнивать **изменение модели на одном корпусе**, а не вычитать public F1 из v2 F1 как эффект обучения.
- Public validation имеет слабую автоматическую разметку; v2 test синтетический, без независимой ручной проверки. Оба результата не являются оценкой закрытого лидерборда.
- После этого запуска v2 test считается использованным для анализа. Следующие настройки смеси/порогов проводить на public calibration + v2 validation, затем проверять на **новом** нетронутом наборе или фиксировать, что test перестал быть независимым.
- Не развёртывать экспериментальную модель вместо `/srv/Hakaton/models/baseline/tfidf_baseline.joblib` без отдельного решения и проверки API: текущий `generator.py` не поддерживает целевой `mcId=101` при `generate_text=true`.
